In [1]:
!pip install imbalanced-learn --quiet

In [2]:
import os, glob, gc
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from imblearn.over_sampling import SMOTENC
from collections import defaultdict

input_folder  = '/content/drive/MyDrive/final_dataset/iot_cleaned_dedup'
output_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)'
bucket_dir    = '/content/buckets_70_30'
target_col    = 'label'
DOWN_TARGET   = 70_000
UP_TARGET     = 30_000

os.makedirs(output_folder, exist_ok=True)
os.makedirs(bucket_dir, exist_ok=True)
all_files = sorted(glob.glob(os.path.join(input_folder, "*.csv")))
print(f"Found {len(all_files)} files.")

# ── Binary/categorical columns that SMOTENC must NOT interpolate ──────────────
# These are identified from CICIoT2023 feature definitions:
# Protocol indicators (0/1), flag values (0/1)
# We will determine indices after loading the first file
CATEGORICAL_FEATURE_NAMES = [
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number',
    'http', 'https', 'dns', 'telnet', 'smtp', 'ssh', 'irc',
    'tcp', 'udp', 'dhcp', 'arp', 'icmp', 'igmp', 'ipv', 'llc',
]

print(f"Output folder : {output_folder}")
print(f"DOWN_TARGET   : {DOWN_TARGET:,}")
print(f"UP_TARGET     : {UP_TARGET:,}")
print(f"Categorical features for SMOTENC: {len(CATEGORICAL_FEATURE_NAMES)}")

Found 62 files.
Output folder : /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)
DOWN_TARGET   : 70,000
UP_TARGET     : 30,000
Categorical features for SMOTENC: 22


In [3]:
bucket_buffers = defaultdict(list)
FLUSH_EVERY    = 200_000
label_to_file  = {}

def get_bucket_path(label):
    if label not in label_to_file:
        idx  = len(label_to_file)
        safe = f"class_{idx:03d}.parquet"
        label_to_file[label] = os.path.join(bucket_dir, safe)
    return label_to_file[label]

def flush_to_disk(label, chunks):
    path   = get_bucket_path(label)
    new_df = pd.concat(chunks, ignore_index=True)
    if os.path.exists(path):
        existing = pd.read_parquet(path)
        new_df   = pd.concat([existing, new_df], ignore_index=True)
    new_df.to_parquet(path, index=False)

print("Pass 1: Bucketing rows by class...")
for file in tqdm(all_files, desc="Reading CSVs"):
    df = pd.read_csv(file)
    for label, group in df.groupby(target_col):
        bucket_buffers[label].append(group)
        total_buffered = sum(len(g) for g in bucket_buffers[label])
        if total_buffered >= FLUSH_EVERY:
            flush_to_disk(label, bucket_buffers[label])
            bucket_buffers[label] = []
    del df
    gc.collect()

print("Flushing remaining buffers...")
for label, chunks in bucket_buffers.items():
    if chunks:
        flush_to_disk(label, chunks)
bucket_buffers.clear()
print(f"\nBucketed {len(label_to_file)} classes.")

Pass 1: Bucketing rows by class...


Reading CSVs:   0%|          | 0/62 [00:00<?, ?it/s]

Flushing remaining buffers...

Bucketed 34 classes.


In [4]:
print("\nPass 2: Balancing each class with SMOTENC (memory-optimised)...")

for label, bucket_path in tqdm(label_to_file.items(), desc="Balancing Classes"):

    # ── Load and immediately downcast to save RAM ─────────────────────────────
    class_df      = pd.read_parquet(bucket_path)
    current_count = len(class_df)
    actual_label  = class_df[target_col].iloc[0]
    print(f"\n  '{actual_label}': {current_count:,} rows")

    # ── BENIGN: keep as-is ────────────────────────────────────────────────────
    if str(actual_label).upper() == 'BENIGN':
        final_df = class_df.copy()
        print(f"  → Kept as-is (BENIGN, {current_count:,} rows)")

    # ── CASE 1: Downsample ────────────────────────────────────────────────────
    elif current_count >= DOWN_TARGET:
        # Sample BEFORE doing anything else — reduces RAM immediately
        final_df = class_df.sample(n=DOWN_TARGET, random_state=42).copy()
        del class_df
        gc.collect()
        class_df = final_df
        print(f"  → Downsampled to {DOWN_TARGET:,}")

    # ── CASE 2: Too few for SMOTENC ───────────────────────────────────────────
    elif current_count < 6:
        final_df = class_df.sample(n=UP_TARGET, replace=True, random_state=42)
        print(f"  → Random oversample to {UP_TARGET:,} (too few for SMOTENC)")

    # ── CASE 3: SMOTENC ───────────────────────────────────────────────────────
    else:
        feature_cols = [c for c in class_df.columns if c != target_col]

        X_real = class_df[feature_cols].select_dtypes(include=[np.number]).copy()

        # Free class_df immediately — we have what we need
        del class_df
        gc.collect()

        X_real.replace([np.inf, -np.inf], np.nan, inplace=True)
        X_real.fillna(X_real.median(), inplace=True)

        # Aggressive downcasting — float32 uses half the RAM of float64
        # Categorical columns → int8 (only 0 or 1, needs 1 byte not 8)
        for col in X_real.columns:
            if col in CATEGORICAL_FEATURE_NAMES:
                X_real[col] = X_real[col].round().clip(0, 1).astype(np.int8)
            else:
                X_real[col] = X_real[col].clip(
                    lower=np.finfo(np.float32).min / 2,
                    upper=np.finfo(np.float32).max / 2
                ).astype(np.float32)

        # If class is large, subsample before SMOTE to reduce RAM
        # We only need enough real samples to generate UP_TARGET synthetic ones
        MAX_REAL_FOR_SMOTE = 5000
        if len(X_real) > MAX_REAL_FOR_SMOTE:
            X_real = X_real.sample(n=MAX_REAL_FOR_SMOTE, random_state=42)
            print(f"    Subsampled to {MAX_REAL_FOR_SMOTE} real samples for SMOTENC")

        cat_indices = [
            i for i, col in enumerate(X_real.columns)
            if col in CATEGORICAL_FEATURE_NAMES
        ]

        k_neighbors = min(5, len(X_real) - 1)

        dummy_df = X_real.sample(
            n=min(k_neighbors + 1, len(X_real)), random_state=0
        ).copy()

        X_combined = pd.concat([X_real, dummy_df], ignore_index=True)
        y_combined  = pd.Series(
            [actual_label] * len(X_real) + ['__DUMMY__'] * len(dummy_df)
        )

        # Free originals immediately before SMOTENC allocates
        del X_real, dummy_df
        gc.collect()

        try:
            sm = SMOTENC(
                categorical_features=cat_indices,
                sampling_strategy={actual_label: UP_TARGET},
                k_neighbors=k_neighbors,
                random_state=42
            )
            X_res, y_res = sm.fit_resample(X_combined, y_combined)
            del X_combined, y_combined, sm
            gc.collect()

            final_df = X_res[y_res == actual_label].copy()
            del X_res, y_res
            gc.collect()

            final_df[target_col] = actual_label

            # Snap binary features back to 0/1
            for col in final_df.columns:
                if col in CATEGORICAL_FEATURE_NAMES:
                    final_df[col] = final_df[col].round().clip(0, 1).astype(np.int8)

            print(f"  → SMOTENC upsampled to {len(final_df):,}")

        except Exception as e:
            print(f"  ⚠ SMOTENC failed ({e}), falling back to random oversample.")
            del X_combined, y_combined
            gc.collect()
            # Reload from parquet since we deleted class_df
            class_df = pd.read_parquet(bucket_path)
            final_df = class_df.sample(n=UP_TARGET, replace=True, random_state=42)
            del class_df
            gc.collect()

    # ── Save immediately and free memory ─────────────────────────────────────
    safe_label = str(actual_label).replace(' ', '_').replace('/', '-')
    out_path   = os.path.join(output_folder, f"{safe_label}.csv")
    final_df.to_csv(out_path, index=False)
    print(f"  ✅ Saved → {out_path}")

    del final_df
    gc.collect()

    # Force a more thorough cleanup between classes
    import ctypes
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

print(f"\n✅ Done! Output: {output_folder}")


Pass 2: Balancing each class with SMOTENC (memory-optimised)...


Balancing Classes:   0%|          | 0/34 [00:00<?, ?it/s]


  'DDOS-ICMP_FLOOD': 2,020,984 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/DDOS-ICMP_FLOOD.csv

  'DDOS-UDP_FLOOD': 1,967,818 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/DDOS-UDP_FLOOD.csv

  'DDOS-PSHACK_FLOOD': 1,730,270 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/DDOS-PSHACK_FLOOD.csv

  'DDOS-SYN_FLOOD': 1,827,885 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/DDOS-SYN_FLOOD.csv

  'DDOS-TCP_FLOOD': 1,611,482 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/DDOS-TCP_FLOOD.csv

  'DOS-UDP_FLOOD': 1,769,959 rows
  → Downsampled to 70,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc

In [5]:
import pandas as pd
import glob
import os

balanced_folder   = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)'
all_balanced_files = glob.glob(os.path.join(balanced_folder, "*.csv"))

distribution = {}
print("Calculating final distribution...")
for file in all_balanced_files:
    df_temp = pd.read_csv(file)
    label   = df_temp['label'].iloc[0] if not df_temp.empty else os.path.basename(file)
    distribution[label] = len(df_temp)

dist_series = pd.Series(distribution).sort_values(ascending=False)
print("\n" + "="*40)
print("FINAL CLASS DISTRIBUTION")
print("="*40)
print(dist_series.to_string())
print("="*40)
print(f"Total rows: {dist_series.sum():,}")

Calculating final distribution...

FINAL CLASS DISTRIBUTION
BENIGN                     1026628
DDOS-ICMP_FLOOD              70000
DDOS-UDP_FLOOD               70000
DDOS-PSHACK_FLOOD            70000
DDOS-TCP_FLOOD               70000
DDOS-SYN_FLOOD               70000
DDOS-RSTFINFLOOD             70000
DDOS-SYNONYMOUSIP_FLOOD      70000
DOS-TCP_FLOOD                70000
DOS-UDP_FLOOD                70000
DOS-SYN_FLOOD                70000
MIRAI-GREETH_FLOOD           70000
MIRAI-UDPPLAIN               70000
MIRAI-GREIP_FLOOD            70000
DDOS-ICMP_FRAGMENTATION      70000
VULNERABILITYSCAN            70000
DDOS-ACK_FRAGMENTATION       70000
DDOS-UDP_FRAGMENTATION       70000
MITM-ARPSPOOFING             70000
DNS_SPOOFING                 70000
RECON-OSSCAN                 70000
RECON-HOSTDISCOVERY          70000
RECON-PORTSCAN               70000
COMMANDINJECTION             30000
BROWSERHIJACKING             30000
BACKDOOR_MALWARE             30000
DICTIONARYBRUTEFORCE         3

In [6]:
import pandas as pd
import glob
import os
import gc

balanced_folder    = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)'
all_balanced_files = glob.glob(os.path.join(balanced_folder, "*.csv"))
all_balanced_files = [f for f in all_balanced_files if 'merged' not in f]

BINARY_FEATS = [
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number', 'http', 'https', 'dns', 'telnet',
    'smtp', 'ssh', 'irc', 'tcp', 'udp', 'dhcp', 'arp',
    'icmp', 'igmp', 'ipv', 'llc'
]

print("Binary feature integrity check (should all be 0 or 1 only)")
print("="*65)
print(f"{'Class':<35} {'Non-binary values found'}")
print("-"*65)

all_clean = True
for file in sorted(all_balanced_files):
    df    = pd.read_csv(file)
    label = df['label'].iloc[0]
    issues = []
    for feat in BINARY_FEATS:
        if feat in df.columns:
            unique_vals = df[feat].unique()
            bad = [v for v in unique_vals if v not in [0, 1, 0.0, 1.0]]
            if bad:
                issues.append(f"{feat}:{len(bad)} bad vals")
    if issues:
        print(f"  {str(label)[:35]:<35} ⚠️  {issues}")
        all_clean = False
    else:
        print(f"  {str(label)[:35]:<35} ✅ clean")
    del df
    gc.collect()

if all_clean:
    print("\n✅ All binary features are clean — SMOTENC worked correctly.")
else:
    print("\n⚠️  Some binary features still have non-binary values.")
    print("   Check the snapping logic in Cell 4.")

Binary feature integrity check (should all be 0 or 1 only)
Class                               Non-binary values found
-----------------------------------------------------------------
  BACKDOOR_MALWARE                    ✅ clean
  BENIGN                              ⚠️  ['fin_flag_number:15 bad vals', 'syn_flag_number:14 bad vals', 'rst_flag_number:9 bad vals', 'psh_flag_number:25 bad vals', 'ack_flag_number:27 bad vals', 'ece_flag_number:5 bad vals', 'cwr_flag_number:4 bad vals', 'http:20 bad vals', 'https:30 bad vals', 'dns:17 bad vals', 'ssh:5 bad vals', 'irc:1 bad vals', 'tcp:27 bad vals', 'udp:24 bad vals', 'dhcp:8 bad vals', 'arp:17 bad vals', 'icmp:12 bad vals', 'igmp:2 bad vals', 'ipv:17 bad vals', 'llc:17 bad vals']
  BROWSERHIJACKING                    ✅ clean
  COMMANDINJECTION                    ✅ clean
  DDOS-ACK_FRAGMENTATION              ⚠️  ['fin_flag_number:36 bad vals', 'syn_flag_number:80 bad vals', 'rst_flag_number:149 bad vals', 'psh_flag_number:96 bad vals', 'ac

In [1]:
import pandas as pd
import glob
import os
import gc

balanced_folder    = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)'
all_balanced_files = glob.glob(os.path.join(balanced_folder, "*.csv"))
all_balanced_files = [f for f in all_balanced_files if 'merged' not in f]

print(f"{'Class Label':<30} | {'Total':^10} | {'Unique':^10} | {'Dupes':^10}")
print("-"*68)

for file in sorted(all_balanced_files):
    df        = pd.read_csv(file)
    label     = df['label'].iloc[0] if not df.empty else os.path.basename(file)
    total     = len(df)
    unique    = len(df.drop_duplicates())
    dupes     = total - unique
    print(f"{str(label)[:30]:<30} | {total:^10,} | {unique:^10,} | {dupes:^10,}")
    del df
    gc.collect()

Class Label                    |   Total    |   Unique   |   Dupes   
--------------------------------------------------------------------
BACKDOOR_MALWARE               |   30,000   |   29,986   |     14    
BENIGN                         | 1,026,628  | 1,026,168  |    460    
BROWSERHIJACKING               |   30,000   |   29,926   |     74    
COMMANDINJECTION               |   30,000   |   29,861   |    139    
DDOS-ACK_FRAGMENTATION         |   70,000   |   69,986   |     14    
DDOS-HTTP_FLOOD                |   30,000   |   30,000   |     0     
DDOS-ICMP_FLOOD                |   70,000   |   69,806   |    194    
DDOS-ICMP_FRAGMENTATION        |   70,000   |   69,992   |     8     
DDOS-PSHACK_FLOOD              |   70,000   |   69,703   |    297    
DDOS-RSTFINFLOOD               |   70,000   |   69,524   |    476    
DDOS-SLOWLORIS                 |   30,000   |   30,000   |     0     
DDOS-SYNONYMOUSIP_FLOOD        |   70,000   |   69,500   |    500    
DDOS-SYN_FLOOD       

In [ ]:
import pandas as pd
import glob
import os
import gc
import numpy as np

balanced_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)'
output_path     = os.path.join(balanced_folder, 'merged.csv')

all_files = glob.glob(os.path.join(balanced_folder, "*.csv"))
all_files = [f for f in all_files if os.path.basename(f) != 'merged.csv']
print(f"Found {len(all_files)} class files to merge.")

dfs = []
for file in all_files:
    df = pd.read_csv(file)
    dfs.append(df)
    print(f"  Loaded: {os.path.basename(file)} → {len(df):,} rows")
    gc.collect()

print("\nConcatenating all classes...")
merged = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"Total rows before shuffle: {len(merged):,}")

print("\nShuffling (pass 1/3)...")
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)
print("Shuffling (pass 2/3)...")
merged = merged.sample(frac=1, random_state=np.random.randint(0, 99999)).reset_index(drop=True)
print("Shuffling (pass 3/3)...")
merged = merged.sample(frac=1, random_state=np.random.randint(0, 99999)).reset_index(drop=True)

print(f"\nFinal label distribution:\n{merged['label'].value_counts().to_string()}")
print(f"\nTotal rows: {len(merged):,}")
print(f"\nSaving to {output_path} ...")
merged.to_csv(output_path, index=False)
print("✅ Done! Merged and shuffled CSV saved.")

Found 34 class files to merge.
  Loaded: DDOS-ICMP_FLOOD.csv → 70,000 rows
  Loaded: DDOS-UDP_FLOOD.csv → 70,000 rows
  Loaded: DDOS-PSHACK_FLOOD.csv → 70,000 rows
  Loaded: DDOS-SYN_FLOOD.csv → 70,000 rows
  Loaded: DDOS-TCP_FLOOD.csv → 70,000 rows
  Loaded: DOS-UDP_FLOOD.csv → 70,000 rows
  Loaded: DDOS-RSTFINFLOOD.csv → 70,000 rows
  Loaded: DDOS-SYNONYMOUSIP_FLOOD.csv → 70,000 rows
  Loaded: DOS-TCP_FLOOD.csv → 70,000 rows
  Loaded: DOS-SYN_FLOOD.csv → 70,000 rows
  Loaded: BENIGN.csv → 1,026,628 rows
  Loaded: MIRAI-GREETH_FLOOD.csv → 70,000 rows
  Loaded: MIRAI-UDPPLAIN.csv → 70,000 rows
  Loaded: MIRAI-GREIP_FLOOD.csv → 70,000 rows
  Loaded: DDOS-ICMP_FRAGMENTATION.csv → 70,000 rows
  Loaded: VULNERABILITYSCAN.csv → 70,000 rows
  Loaded: DDOS-ACK_FRAGMENTATION.csv → 70,000 rows
  Loaded: DDOS-UDP_FRAGMENTATION.csv → 70,000 rows
  Loaded: MITM-ARPSPOOFING.csv → 70,000 rows
  Loaded: BACKDOOR_MALWARE.csv → 30,000 rows
  Loaded: BROWSERHIJACKING.csv → 30,000 rows
  Loaded: COMMANDI